# Server-Side Agent Routing Lab

**Duration:** ~30 minutes  
**Scenario:** You're building a server-side routing system where a single Supervisor Agent receives all user requests and delegates them to the correct specialist agent via stored procedures. This eliminates cross-surface routing variance — every client gets identical behavior.

**What you'll do:**
1. Verify the routing infrastructure (agents + procedures)
2. Test routing by sending domain-specific queries to the supervisor
3. Test edge cases with ambiguous queries
4. Run a formal evaluation using Cortex Agent Evaluations
5. Inspect per-query results and identify routing failures
6. Understand how to iterate and improve routing quality

**Prerequisites:** Run `setup.sql` before starting this notebook. It creates the database, warehouse, agents, procedures, and evaluation data.

---
## Section 1: Connect & Verify

First, establish a session and confirm that all infrastructure objects from `setup.sql` exist.

In [ ]:
# Connection setup — works in both Snowsight notebooks and local Jupyter
import os

try:
    # Snowsight notebook: session already exists
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    # Local Jupyter: create session from environment or connection config
    from snowflake.snowpark import Session
    connection_params = {
        "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
        "user": os.environ.get("SNOWFLAKE_USER"),
        "password": os.environ.get("SNOWFLAKE_PASSWORD"),
        "role": os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN"),
        "warehouse": "ROUTING_DEMO_WH",
        "database": "ROUTING_DEMO",
        "schema": "ROUTING",
    }
    session = Session.builder.configs(connection_params).create()

# Set context
session.sql("USE DATABASE ROUTING_DEMO").collect()
session.sql("USE SCHEMA ROUTING").collect()
session.sql("USE WAREHOUSE ROUTING_DEMO_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Database: {session.sql('SELECT CURRENT_DATABASE()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

In [ ]:
# Verify agents exist
agents = session.sql("SHOW AGENTS IN SCHEMA ROUTING_DEMO.ROUTING").collect()
print("Agents:")
for row in agents:
    print(f"  - {row['name']}")

print()

# Verify procedures exist
procs = session.sql("SHOW PROCEDURES IN SCHEMA ROUTING_DEMO.ROUTING").collect()
print("Procedures:")
for row in procs:
    print(f"  - {row['name']}({row['arguments']})")

---
## Section 2: Test Routing

Send three queries to the Supervisor — one per specialist domain. The supervisor should route each to the correct specialist. We call `DATA_AGENT_RUN` on the supervisor, which internally picks a tool (specialist) based on its orchestration instructions.

**Expected routing:**
- Sales question → `sales_agent` tool → `RUN_SALES_AGENT` procedure
- Product question → `product_agent` tool → `RUN_PRODUCT_AGENT` procedure
- Supply chain question → `supply_chain_agent` tool → `RUN_SUPPLY_CHAIN_AGENT` procedure

In [ ]:
import json

test_queries = [
    ("Sales", "What was our Q1 revenue and how does pipeline look for Q2?"),
    ("Product", "A customer reports the motor overheating after 10 minutes of use"),
    ("Supply Chain", "What's the current inventory level for SKU-4521 and expected restock date?"),
]

for expected_domain, query in test_queries:
    body = json.dumps({
        "messages": [{"role": "user", "content": [{"type": "text", "text": query}]}]
    })
    escaped = body.replace("'", "''")
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
            'ROUTING_DEMO.ROUTING.SUPERVISOR',
            '{escaped}'
        )
    """).collect()

    response = str(result[0][0]) if result else 'No response'
    print(f"[Expected: {expected_domain}]")
    print(f"Q: {query}")
    print(f"A: {response[:400]}")
    print("=" * 70)
    print()

**Check the output above.** Each response should come from the expected specialist domain. If routing is correct, you'll see:
- Sales-related content for the first query
- Troubleshooting steps for the second query
- Inventory/logistics data for the third query

---
## Section 3: Test Edge Cases

Ambiguous queries are where routing gets interesting. The supervisor must reason about which specialist is the best fit. Let's test a query that could plausibly go to either supply chain or product support.

In [ ]:
# Ambiguous query: delivery delay could be supply chain (logistics) or product support (customer issue)
ambiguous_query = "A customer complained about delivery delays on their recent purchase"

body = json.dumps({
    "messages": [{"role": "user", "content": [{"type": "text", "text": ambiguous_query}]}]
})
escaped = body.replace("'", "''")
result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
        'ROUTING_DEMO.ROUTING.SUPERVISOR',
        '{escaped}'
    )
""").collect()

response = str(result[0][0]) if result else 'No response'
print(f"Q: {ambiguous_query}")
print(f"A: {response[:500]}")
print()
print("Note: This likely routes to supply_chain_agent (delivery/logistics).")
print("If you want it routed differently, adjust the orchestration keywords in the supervisor spec.")

**Key insight:** Ambiguous queries reveal where your orchestration instructions need refinement. If the supervisor routes incorrectly, the fix is to:
1. Add clearer routing keywords to the orchestration instructions
2. Improve tool descriptions to be more discriminating
3. Add the ambiguous case to your evaluation dataset with the expected routing

---
## Section 4: Run Evaluation

Now we validate routing quality systematically using Cortex Agent Evaluations. The `setup.sql` script already created:
- `EVAL_DATA` table with 6 ground-truth rows (2 per domain)
- `EVAL_STAGE` for the evaluation config YAML

We'll create the evaluation dataset, upload the config, and launch an evaluation run.

In [ ]:
# Create the evaluation dataset from our EVAL_DATA table
session.sql("""
    CALL SYSTEM$CREATE_EVALUATION_DATASET(
        'Cortex Agent',
        'ROUTING_DEMO.ROUTING.EVAL_DATA',
        'ROUTING_DEMO.ROUTING.ROUTING_EVAL_DATASET',
        OBJECT_CONSTRUCT('query_text', 'INPUT_QUERY', 'expected_tools', 'GROUND_TRUTH')
    )
""").collect()
print("Evaluation dataset ROUTING_EVAL_DATASET created.")

In [ ]:
import tempfile, os

# Upload eval config YAML to stage
eval_yaml = """evaluation:
  agent_params:
    agent_name: "ROUTING_DEMO.ROUTING.SUPERVISOR"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "Routing quality baseline"
    description: "Validates that the supervisor routes to the correct specialist agent"
  source_metadata:
    type: "dataset"
    dataset_name: "ROUTING_DEMO.ROUTING.ROUTING_EVAL_DATASET"

metrics:
  - "answer_correctness"
  - "tool_selection_accuracy"
  - "logical_consistency"
"""

tmp_path = os.path.join(tempfile.gettempdir(), 'eval_config.yaml')
with open(tmp_path, 'w') as f:
    f.write(eval_yaml)

session.sql(f"PUT 'file://{tmp_path}' @ROUTING_DEMO.ROUTING.EVAL_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE").collect()
os.unlink(tmp_path)
print("Eval config uploaded to @ROUTING_DEMO.ROUTING.EVAL_STAGE/eval_config.yaml")

In [ ]:
# Start the evaluation run
result = session.sql("""
    CALL EXECUTE_AI_EVALUATION(
        'START',
        OBJECT_CONSTRUCT('run_name', 'routing-baseline-1'),
        '@ROUTING_DEMO.ROUTING.EVAL_STAGE/eval_config.yaml'
    )
""").collect()
print("Evaluation started:", result[0][0])

In [ ]:
# Check evaluation status (re-run this cell until status shows COMPLETE)
import time

for attempt in range(12):
    status = session.sql("""
        CALL EXECUTE_AI_EVALUATION(
            'STATUS',
            OBJECT_CONSTRUCT('run_name', 'routing-baseline-1'),
            '@ROUTING_DEMO.ROUTING.EVAL_STAGE/eval_config.yaml'
        )
    """).collect()
    status_str = str(status[0][0])
    print(f"Attempt {attempt+1}: {status_str}")
    if 'COMPLETE' in status_str.upper() or 'FAILED' in status_str.upper():
        break
    time.sleep(30)

print("\nFinal status:", status_str)

---
## Section 5: Inspect Results

Once the evaluation completes, query the results to see per-metric averages and per-query details. The key metric for routing is `tool_selection_accuracy` — did the supervisor pick the right specialist?

In [ ]:
# Average score per metric
results = session.sql("""
    SELECT 
        METRIC_NAME,
        ROUND(AVG(EVAL_AGG_SCORE), 3) AS AVG_SCORE,
        COUNT(*) AS NUM_RECORDS
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'ROUTING_DEMO',
        'ROUTING',
        'SUPERVISOR',
        'CORTEX AGENT',
        'routing-baseline-1'
    ))
    GROUP BY METRIC_NAME
    ORDER BY METRIC_NAME
""").collect()

print(f"{'Metric':<30} {'Avg Score':<12} {'Records'}")
print("-" * 55)
for row in results:
    print(f"{row['METRIC_NAME']:<30} {row['AVG_SCORE']:<12} {row['NUM_RECORDS']}")

In [ ]:
# Per-query detail for tool_selection_accuracy — identify any routing failures
details = session.sql("""
    SELECT 
        INPUT,
        EVAL_AGG_SCORE,
        DURATION_MS,
        LEFT(OUTPUT, 200) AS RESPONSE_PREVIEW
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'ROUTING_DEMO',
        'ROUTING',
        'SUPERVISOR',
        'CORTEX AGENT',
        'routing-baseline-1'
    ))
    WHERE METRIC_NAME = 'tool_selection_accuracy'
    ORDER BY EVAL_AGG_SCORE ASC
""").collect()

print("Tool Selection Accuracy — Per Query")
print("=" * 70)
for row in details:
    score = row['EVAL_AGG_SCORE']
    status = '✓' if score >= 0.8 else '✗'
    print(f"{status} Score: {score:.2f} | Duration: {row['DURATION_MS']}ms")
    print(f"  Query: {row['INPUT'][:80]}")
    print(f"  Response: {row['RESPONSE_PREVIEW'][:100]}...")
    print()

---
## Section 6: Iterate & Improve

If any queries scored below 1.0 on `tool_selection_accuracy`, here's how to improve:

### 1. Refine Orchestration Instructions
Add more specific routing keywords to the supervisor's `orchestration` instructions. For example, if "delivery delays" was misrouted, add "delivery" explicitly to the supply chain bullet.

### 2. Improve Tool Descriptions
Make tool descriptions more discriminating. If two tools overlap on a concept, add exclusion language (e.g., "NOT for customer-facing complaints").

### 3. Expand the Evaluation Dataset
Add ambiguous queries with their expected routing as ground truth. This makes boundary cases measurable.

### 4. Re-run Evaluation
After changes, run a new evaluation with a different `run_name` and compare scores side-by-side.

```sql
-- Example: Update the supervisor's orchestration instructions
CREATE OR REPLACE AGENT ROUTING_DEMO.ROUTING.SUPERVISOR
COMMENT = 'Server-side routing agent — v2 with improved keywords'
FROM SPECIFICATION $$
-- ... updated spec with better routing keywords ...
$$;

-- Re-run evaluation with a new run name
CALL EXECUTE_AI_EVALUATION(
    'START',
    OBJECT_CONSTRUCT('run_name', 'routing-v2'),
    '@ROUTING_DEMO.ROUTING.EVAL_STAGE/eval_config.yaml'
);
```

---
## Summary: What You Accomplished

| Step | What you did |
|------|-------------|
| **Verify** | Confirmed agents and procedures are deployed |
| **Test** | Sent domain-specific queries and validated correct routing |
| **Edge Cases** | Explored ambiguous queries to understand routing boundaries |
| **Evaluate** | Ran a formal evaluation with `tool_selection_accuracy` |
| **Inspect** | Identified per-query scores and any routing failures |
| **Iterate** | Learned how to refine instructions and re-evaluate |

### Key Takeaways

- **Server-side routing** eliminates cross-surface variance — identical behavior everywhere
- **Stored procedure wrappers** make any specialist agent callable as a tool
- **`tool_selection_accuracy`** makes routing quality measurable and regressionable
- **Iteration loop:** change instructions → re-evaluate → compare scores → promote

### What's Next

- Add more specialist agents (expand from 3 to your full roster)
- Expand evaluation coverage with more boundary/ambiguous cases
- Integrate evaluation into CI/CD for automated regression checks
- Expose the supervisor as a single MCP tool for external clients

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS ROUTING_DEMO CASCADE").collect()
# print("Lab resources cleaned up.")